# Preparación de Datos

Finalmente con la herramienta de descarga podemos entrar al siguiente apartado. La preparación de los datos. 

## Agro en acción
Denominando Agro a nuestra herramienta para la descarga de archivos del DGSIAP. Haremos una prueba. Supongamos que queremos la información de la producción agricola municipal de todos los años disponibles

In [1]:
from scripts.download import agro_project

file_tree = agro_project()

Conexion Establecida
Usted escogio Agricola
Los datos historicos se presentan en el espacio denomiado: Información por nivel
Los datos se encuentran divididos en 3 bloques:

1. Municipal
2. Nacional
3. No seguimiento


In [2]:

def load_filetree(ftree):
    for v in ftree.values():
        v['Codebook'].downcheck()
        data_file = v['Data']
        for dv in data_file.values():
            dv.downcheck()

load_filetree(file_tree)


https://nube.agricultura.gob.mx/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7 Diccionario_agricola_2003_a_2025.xlsx /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap/Diccionario_agricola_2003_a_2025.xlsx
El archivo ya fue descargado. Se encuentra en /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap/Diccionario_agricola_2003_a_2025.xlsx
https://nube.agricultura.gob.mx/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025 Cierre_agricola_mun_2025.csv /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap/Cierre_agricola_mun_2025.csv
El archivo ya fue descargado. Se encuentra en /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/dgsiap/Cierre_agricola_mun_2025.csv
https://nube.agricultura.gob.mx/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024 Cierre_agricola_mu

## Lectura de Datos
### Diccionario de Datos

Con los archivos descargados. Procedemos a abrir el diccionario de Datos.

In [3]:
from scripts.config import DATA_FOLDERS
import pandas as pd

print(file_tree)

block = "Municipal"
dic = file_tree['Municipal']['Codebook']
data = file_tree['Municipal']['Data']['2025']


raw_catalog = pd.read_excel(dic.filepath / dic.file)
raw_catalog.head(n=5)

{'Municipal': {'Codebook': <scripts.File.File object at 0x7f8fbe86cec0>, 'Data': {'2025': <scripts.File.File object at 0x7f8fbed7c910>, '2024': <scripts.File.File object at 0x7f8fbed7c7d0>, '2023': <scripts.File.File object at 0x7f8fbe85d220>, '2022': <scripts.File.File object at 0x7f8fbe85d480>, '2021': <scripts.File.File object at 0x7f8fbe9afd10>, '2020': <scripts.File.File object at 0x7f8fbe803460>, '2019': <scripts.File.File object at 0x7f8fbe803570>, '2018': <scripts.File.File object at 0x7f8fbe869850>, '2017': <scripts.File.File object at 0x7f8fbe1a2950>, '2016': <scripts.File.File object at 0x7f8fbe83fc50>, '2015': <scripts.File.File object at 0x7f8fbedd16d0>, '2014': <scripts.File.File object at 0x7f8fbedd4670>, '2013': <scripts.File.File object at 0x7f8fbedd4ad0>, '2012': <scripts.File.File object at 0x7f8fbe1aca10>, '2011': <scripts.File.File object at 0x7f8fbe9774d0>, '2010': <scripts.File.File object at 0x7f8fbe9777d0>, '2009': <scripts.File.File object at 0x7f8fbedc1180>, 

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Como vemos. El diccionario esta mal formateado. Entonces debemos ajustarlo. 

In [4]:
from scripts.format import format_codebook, get_encoding

catalog, variables = format_codebook(raw_catalog)
catalog

,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...


Obteniendo asi el diccionario de datos asociado y las variables involucradas.

### Datos

Ahora vamos a leer los datos. Donde veremos un problema de codificación.

In [5]:
try:
    pd.read_csv(data.filepath / data.file)
except UnicodeDecodeError as e:
    print(e)

'utf-8' codec can't decode byte 0xf1 in position 350: invalid continuation byte


Entonces debemos detectar la codificación de origen y usarlo para leerlo.

In [6]:
file_code = get_encoding(data.filepath / data.file)
print(file_code)

df = pd.read_csv(data.filepath / data.file, encoding=file_code['encoding'], low_memory=False)
df.head(n=5)

{'encoding': 'ISO-8859-1', 'confidence': 0.8167673945426941, 'language': 'es'}


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


# Guardado de Datos.

Finalmente, una vez los datos descargados y formateados debemos guardarlos para su futuro procesamiento. Para mejor sintonia con el resto de pasos. Usaremos la petición para darles nombres a los archivos. Para ello debemos generalizar el encoding de los archivos dentro del arbol.

In [7]:
encode_tree = {}

def encode_block(file_block: dict):
    encode = {"Codebook":None,
             "Data": {}}
     
    encode['Codebook'] = file_block['Codebook']
    encode['Data'] = {year: {"file": file,
                             "encoding": get_encoding(file.ubication)['encoding']} for year, file in file_block['Data'].items()}
    return encode

encode_tree['Municipal'] = encode_block(file_tree['Municipal'])

Finalmente vemos el encoding requerido de cada archivo. Ahora vamos a guardar la copia en formato excel para garantizar la codificación UTF-8.

In [8]:
print(encode_tree)

{'Municipal': {'Codebook': <scripts.File.File object at 0x7f8fbe86cec0>, 'Data': {'2025': {'file': <scripts.File.File object at 0x7f8fbed7c910>, 'encoding': 'ISO-8859-1'}, '2024': {'file': <scripts.File.File object at 0x7f8fbed7c7d0>, 'encoding': 'ISO-8859-1'}, '2023': {'file': <scripts.File.File object at 0x7f8fbe85d220>, 'encoding': 'ISO-8859-1'}, '2022': {'file': <scripts.File.File object at 0x7f8fbe85d480>, 'encoding': 'ISO-8859-1'}, '2021': {'file': <scripts.File.File object at 0x7f8fbe9afd10>, 'encoding': 'ISO-8859-1'}, '2020': {'file': <scripts.File.File object at 0x7f8fbe803460>, 'encoding': 'ISO-8859-1'}, '2019': {'file': <scripts.File.File object at 0x7f8fbe803570>, 'encoding': 'ISO-8859-1'}, '2018': {'file': <scripts.File.File object at 0x7f8fbe869850>, 'encoding': 'ISO-8859-1'}, '2017': {'file': <scripts.File.File object at 0x7f8fbe1a2950>, 'encoding': 'ISO-8859-1'}, '2016': {'file': <scripts.File.File object at 0x7f8fbe83fc50>, 'encoding': 'ISO-8859-1'}, '2015': {'file': <

Aprovecharemos la clase FIle previamente creada para facilitar los nombres.

In [10]:
from scripts.File import File

block = "Municipal"

def formatted_data(format_block, block):
    formatted = {"Codebook": {}, "Data": {}}
    codebook = format_block['Codebook']
    file_code = pd.read_excel(codebook.filepath / codebook.file)
    catalog, variables = format_codebook(file_code)
    catfile = File(DATA_FOLDERS['interm'] / codebook.source, f"Diccionario_Datos_{block}.xlsx", codebook.source)
    varfile = File(DATA_FOLDERS['interm'] / codebook.source, f"Tipo_Variables_{block}.xlsx", codebook.source)
    formatted['Codebook'] = {"Diccionario": catfile, "Variables": varfile}
    catalog.to_excel(catfile.get_local_ubication())
    variables.to_excel(varfile.get_local_ubication())
    data = format_block['Data']
    for y, enficode in data.items():
        file, encode = enficode.values()
        dfile = File(DATA_FOLDERS['interm'] / codebook.source, file.file, file.source)
        dset = pd.read_csv(file.ubication, encoding= encode, low_memory=False)
        dset.to_excel(f"{dfile.get_local_ubication(ext=0)}.xlsx")
        formatted['Data'][y] = dfile
    return formatted

formatted_data(encode_tree['Municipal'], 'Municipal')

{'Codebook': {'Diccionario': <scripts.File.File at 0x7f8f7aec3da0>,
  'Variables': <scripts.File.File at 0x7f8f75ef2180>},
 'Data': {'2025': <scripts.File.File at 0x7f8f75ef20f0>,
  '2024': <scripts.File.File at 0x7f8f7aeffe30>,
  '2023': <scripts.File.File at 0x7f8f7aeff770>,
  '2022': <scripts.File.File at 0x7f8f7aeff890>,
  '2021': <scripts.File.File at 0x7f8f7aeff2f0>,
  '2020': <scripts.File.File at 0x7f8f7aefef90>,
  '2019': <scripts.File.File at 0x7f8f75ef08c0>,
  '2018': <scripts.File.File at 0x7f8f75ef1010>,
  '2017': <scripts.File.File at 0x7f8f75ef10a0>,
  '2016': <scripts.File.File at 0x7f8f75ef1910>,
  '2015': <scripts.File.File at 0x7f8f75ef0830>,
  '2014': <scripts.File.File at 0x7f8f75ef0f80>,
  '2013': <scripts.File.File at 0x7f8f75ef1760>,
  '2012': <scripts.File.File at 0x7f8f75ef07a0>,
  '2011': <scripts.File.File at 0x7f8f75ef0710>,
  '2010': <scripts.File.File at 0x7f8f75ef0950>,
  '2009': <scripts.File.File at 0x7f8f75ef1370>,
  '2008': <scripts.File.File at 0x7f